In [1]:
!pip install pandas

**Cargar datos desde GitHub**

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/kevinnumanzor17/etl-seguros-pipeline/refs/heads/main/data/raw/siniestros.csv"

df = pd.read_csv(url)

df.head()

,id_siniestro,id_poliza,fecha_siniestro,monto_siniestro,estado
0,1,17400,16-10-2025,2092.59,ABIERTO
1,2,2465,01/08/2025,"7,076.25",Abierto
2,3,15785,19-09-2025,702.27,cerrado
3,4,14299,27/09/2025,274.63,Abierto
4,5,12908,01/12/2025,"9,377.69",Rechazado


**Exploración**

In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4620 entries, 0 to 4619
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_siniestro     4620 non-null   int64 
 1   id_poliza        4620 non-null   int64 
 2   fecha_siniestro  4620 non-null   object
 3   monto_siniestro  4004 non-null   object
 4   estado           3322 non-null   object
dtypes: int64(2), object(3)
memory usage: 180.6+ KB


,0
id_siniestro,0
id_poliza,0
fecha_siniestro,0
monto_siniestro,616
estado,1298


**Limpieza de datos**

In [4]:
siniestros = df.copy()

# limpiar nombres de columnas
siniestros.columns = siniestros.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in siniestros.select_dtypes(include='object').columns:
    siniestros[col] = siniestros[col].astype(str).str.strip()

# convertir vacíos en null
siniestros = siniestros.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
siniestros = siniestros.drop_duplicates()

**convertir fecha**

In [5]:
siniestros['fecha_siniestro'] = pd.to_datetime(
    siniestros['fecha_siniestro'],
    errors='coerce'
)

/tmp/ipykernel_1107/1711211165.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  siniestros['fecha_siniestro'] = pd.to_datetime(


**convertir monto a número**

In [6]:
siniestros['monto_siniestro'] = pd.to_numeric(
    siniestros['monto_siniestro'],
    errors='coerce'
)

**Separar válidos y rechazados**

In [7]:
validos = siniestros[
    siniestros['fecha_siniestro'].notna() &
    siniestros['monto_siniestro'].notna()
].copy()

rechazados = siniestros[
    siniestros['fecha_siniestro'].isna() |
    siniestros['monto_siniestro'].isna()
].copy()

Motivo de rechazo

In [8]:
def motivo(row):
    motivos = []

    if pd.isna(row['fecha_siniestro']):
        motivos.append("fecha_invalida")

    if pd.isna(row['monto_siniestro']):
        motivos.append("monto_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

# **Exportar archivos**

In [9]:
validos.to_csv("siniestros_curated.csv", index=False)
rechazados.to_csv("siniestros_rejects.csv", index=False)